## <center>Khalifa University — Agentic AI Program</center>
### <center>Program 5 · Day 1: Advanced Agentic Workflows</center>
# Exercise 5: Plan and Delegate with an LLM Manager

**Module:** Dynamic Multi-Agent Orchestration, State Machines & Tool Safety  
**Companion Script:** `Day 1/Excercises/day1_workbook_code/patient_manager.py`  
**Estimated Practical Time:** 45–60 minutes  
**Target Model:** Local Ollama (`qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`)

---

### Aim & Learning Objectives
By the end of this exercise, you will be able to:
1. **Implement dynamic orchestration** using a Manager Agent that plans and delegates actions.
2. **Track workflow memory** through an explicit `state` dictionary (`note`, `facts`, `report`, `review`, `error_inserted`).
3. **Enforce deterministic prerequisites** via `allowed_actions(state)` so the LLM cannot take invalid actions.
4. **Constrain manager responses** using a strict JSON schema (`plan`, `action`, `reason`).
5. **Invalidate stale reviews** immediately whenever a draft or facts are modified (`state["review"] = None`).
6. **Enforce a strict step budget** (`MAX_STEPS = 10`) to prevent infinite looping.
7. **Test recovery behavior, budget bounds, and premature save rejection** across clean, error-injected, and step-limited runs.

## Dynamic Orchestration Architecture

Unlike static pipelines (Exercises 1–3) or hardcoded branch loops (Exercise 4), Exercise 5 implements **dynamic agentic orchestration**. A Manager LLM evaluates the system's current state and plans which specialized tool to execute next.

Crucially, **Python acts as the strict gatekeeper**: the LLM is only permitted to choose actions from a Python-enforced whitelist (`allowed_actions(state)`), actions are executed by a controlled dispatcher, and safety invariants (such as stale review invalidation and step budgets) are strictly enforced in code:

<p align="center">
  <img src="images/exercise5_manager_flow.jpg" alt="Exercise 5: Dynamic LLM Manager Orchestration & State Machine" style="max-width: 100%; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
</p>

### State Machine Transition Rules & Prerequisite Matrix

| Current State Condition | Allowed Actions | Strictly Blocked Actions | State Modification Rule | Safety Justification |
|---|---|---|---|---|
| `state["note"] == ""` | `["load", "escalate"]` | extract, write, review, revise, save | Loads note text into state | Cannot extract clinical facts from an empty source. |
| `state["facts"] == ""` | `["extract", "escalate"]` | write, review, revise, save | Populates `state["facts"]`, clears `state["report"]` | Cannot write clinical report without verified facts. |
| `state["report"] == ""` | `["write", "escalate"]` | review, revise, save | Populates `state["report"]`, sets `review = None` | Cannot review a non-existent report draft. |
| `state["review"] is None` | `["review", "escalate"]` | revise, save | Populates `state["review"]` with PASS or REVISE | Cannot revise or save without a fresh, valid review. |
| `review["decision"] == "PASS"` | `["save", "escalate"]` | revise, write | Invokes `save_report()` tool to persist output | **Approved:** draft is verified and ready for clinician review. |
| `review["decision"] == "REVISE"` | `["revise", "extract", "escalate"]` | save, review | Updates report, **sets `review = None`** | **CRITICAL:** Stale review is invalidated; save is impossible. |

In [1]:
import argparse
import json
import os
import sys
import time
from pathlib import Path
from uuid import uuid4
from ollama import chat

MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b-instruct")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
CONTEXT_TOKENS = 8192
TEMPERATURE = 0
CALL_COUNT = 0
MAX_STEPS = 10

print(f"Manager Configuration:")
print(f"  Target Model: {MODEL_NAME}")
print(f"  Step Budget:  {MAX_STEPS}")

Manager Configuration:
  Target Model: qwen3:4b-instruct
  Step Budget:  10


## Preflight Check & Note Fixture Setup
Verifies Ollama connectivity and ensures `fictional_patient_note.txt` is present.

In [2]:
import importlib.util
import json
import urllib.request
import urllib.error

class PreflightError(RuntimeError):
    pass

def run_preflight():
    # 1. Check Python library
    if importlib.util.find_spec("ollama") is None:
        raise PreflightError("The 'ollama' library is missing. Install with: pip install ollama")
    
    # 2. Check Ollama API reachability
    try:
        req = urllib.request.Request(f"{OLLAMA_URL}/api/tags")
        with urllib.request.urlopen(req, timeout=3) as resp:
            data = json.loads(resp.read().decode())
            installed_models = [m.get("name") for m in data.get("models", [])]
    except Exception as e:
        raise PreflightError(
            f"Cannot connect to Ollama at {OLLAMA_URL}. Ensure Ollama is running (`ollama serve`). Details: {e}"
        )
    
    print(f"Ollama server is active at {OLLAMA_URL}.")
    print(f"Installed models detected ({len(installed_models)}):")
    for m in installed_models:
        print(f"  - {m}")
    
    # Smart Model Resolution:
    # 1. Respect explicit environment variable if provided
    # 2. Check preferred MODEL_NAME (qwen3:4b-instruct)
    # 3. Check recommended workshop models (qwen3.5:4b, qwen2.5:7b, qwen2.5:3b)
    # 4. Fall back to any installed qwen / available model
    global MODEL_NAME
    env_model = os.environ.get("OLLAMA_MODEL")
    if env_model:
        MODEL_NAME = env_model
        print(f"\nUsing environment specified model: '{MODEL_NAME}'")
        return
    
    if any(m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]) for m in installed_models):
        for m in installed_models:
            if m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]):
                MODEL_NAME = m
                break
        print(f"\nActive model selected: '{MODEL_NAME}' (target match)")
        return
    
    for candidate in ["qwen3.5:4b", "qwen2.5:7b", "qwen2.5:3b"]:
        match = next((m for m in installed_models if m.startswith(candidate) or candidate.startswith(m.split(":")[0])), None)
        if match:
            MODEL_NAME = match
            print(f"\n[NOTICE] '{MODEL_NAME}' selected from installed models (aligns with recommended workshop models).")
            return
            
    if installed_models:
        MODEL_NAME = installed_models[0]
        print(f"\n[NOTICE] Selected first available model: '{MODEL_NAME}'")
    else:
        raise PreflightError("No models found in Ollama. Pull a model with: ollama pull qwen2.5:7b")

try:
    run_preflight()
except PreflightError as err:
    print(f"PREFLIGHT CHECK FAILED: {err}")

# Ensure fictional_patient_note.txt exists in local or candidate paths
candidate_paths = [
    Path("fictional_patient_note.txt"),
    Path("Day 1/Excercises/notebooks/fictional_patient_note.txt"),
    Path("../fictional_patient_note.txt"),
    Path("../../Day 1/Excercises/notebooks/fictional_patient_note.txt"),
]
SOURCE = next((p for p in candidate_paths if p.exists()), None)
if SOURCE is None:
    SOURCE = Path("fictional_patient_note.txt")
    SOURCE.write_text(
        "Patient ID: DEMO-001\n"
        "Age: 60 years\n"
        "Diagnosis recorded by the clinician: Colon cancer\n"
        "Reported symptoms: Abdominal pain and blood in the stool\n"
        "Colonoscopy finding: A mass in the colon\n"
        "Biopsy result: Adenocarcinoma\n"
        "Cancer stage: Not provided\n"
        "Treatment plan: Not provided\n",
        encoding="utf-8"
    )
print(f"Source note verified: {SOURCE} ({SOURCE.stat().st_size} bytes)")

Ollama server is active at http://localhost:11434.
Installed models detected (3):
  - qwen3.5:4b
  - qwen3:4b-instruct
  - qwen2.5:3b

Active model selected: 'qwen3:4b-instruct' (target match)
Source note verified: fictional_patient_note.txt (269 bytes)


## Step 1: Base Worker Functions & Local Tools

Notice the tool safety rules:
- `load_patient_note()`: reads only the fixed, local note file. The LLM never supplies arbitrary paths.
- `save_report()`: raises an error if `state["review"]["decision"] != "PASS"`. Unreviewed or rejected drafts can never be saved.

In [3]:
KNOWN_ERROR = (
    "\nThe patient has stage III cancer. "
    "Chemotherapy is planned."
)

def ask_model(instructions, data, json_output=False):
    global CALL_COUNT
    settings = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user", "content": data},
        ],
        "options": {"temperature": TEMPERATURE, "num_ctx": CONTEXT_TOKENS},
        "stream": False,
    }
    if json_output:
        settings["format"] = "json" if json_output is True else json_output
    CALL_COUNT += 1
    response = chat(**settings)
    return response.message.content

def extract_facts(note):
    return ask_model(
        "Extract patient ID, age, recorded diagnosis, symptoms, "
        "colonoscopy, biopsy, stage, and treatment as bullets. "
        "Use only the note. Write Not provided for missing facts. "
        "Do not infer stage or recommend treatment.",
        note,
    )

def write_report(facts):
    return ask_model(
        "Write a patient summary under 150 words. Include all "
        "eight supplied fields. State when stage and treatment "
        "are not provided. Add no conclusions or recommendations. "
        "Say 'recorded diagnosis'; do not infer how or when the "
        "diagnosis was made or connect it causally to symptoms/tests.",
        facts,
    )

def review_report(note, report):
    answer = ask_model(
        "Compare the report with the original note. Check all "
        "eight fields for omissions, errors, and invented facts. "
        "Reject invented explanations of how or when diagnosis was made. "
        "Return JSON with decision (PASS or REVISE) and issues "
        "(a list of specific correction strings). Use PASS only "
        "if there are no issues, with an empty issues list. "
        "Use REVISE with a nonempty issues list otherwise.",
        f"ORIGINAL NOTE:\n{note}\n\nDRAFT:\n{report}",
        json_output=True,
    )
    return json.loads(answer)

def revise_report(note, report, issues):
    return ask_model(
        "Correct the draft using the original note and feedback. "
        "The note is the source of truth. Include all eight fields. "
        "Do not infer stage or treatment. Return only the corrected "
        "report, under 150 words. Say 'recorded diagnosis'; do not "
        "infer how or when it was made or link it to symptoms/tests.",
        f"NOTE:\n{note}\nDRAFT:\n{report}\n"
        f"CORRECTIONS:\n{json.dumps(issues)}",
    )

def load_patient_note():
    return SOURCE.read_text(encoding="utf-8")

def save_report(state, folder):
    if not state.get("review") or state["review"].get("decision") != "PASS":
        raise ValueError("A reviewed PASS is required before saving.")
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "reviewed_draft.txt"
    path.write_text(
        "CLASSROOM DRAFT - human review required\n\n" + state["report"],
        encoding="utf-8",
    )
    return str(path)

print("Base worker functions and local tools initialized.")

Base worker functions and local tools initialized.


## Step 2: State Prerequisites & Allowed Actions

The function `allowed_actions(state)` acts as a deterministic state machine:
- No note loaded? -> `["load", "escalate"]`
- No facts extracted? -> `["extract", "escalate"]`
- No report drafted? -> `["write", "escalate"]`
- Review is None? -> `["review", "escalate"]`
- Review passed? -> `["save", "escalate"]`
- Review failed? -> `["revise", "extract", "escalate"]`

In [4]:
def allowed_actions(state):
    if not state.get("note"):
        return ["load", "escalate"]
    if not state.get("facts"):
        return ["extract", "escalate"]
    if not state.get("report"):
        return ["write", "escalate"]
    if state.get("review") is None:
        return ["review", "escalate"]
    if state["review"].get("decision") == "PASS":
        return ["save", "escalate"]
    return ["revise", "extract", "escalate"]

# Test empty state
test_state = {"note": "", "facts": "", "report": "", "review": None}
print("Initial allowed actions:", allowed_actions(test_state))

Initial allowed actions: ['load', 'escalate']


## Step 3: Structured Decision Prompting (`choose_next`)

The manager is prompted with the current state, remaining steps, and list of allowed actions. It is forced via JSON Schema to choose **exactly one** action from `allowed`.

In [5]:
def choose_next(state, allowed, steps_left):
    schema = {
        "type": "object",
        "properties": {
            "plan": {"type": "array", "items": {"type": "string"}, "minItems": 1},
            "action": {"type": "string", "enum": allowed},
            "reason": {"type": "string"},
        },
        "required": ["plan", "action", "reason"],
        "additionalProperties": False,
    }
    raw = ask_model(
        "You manage a fictional report workflow. Return JSON with "
        "plan (a short list of task steps), action (one allowed "
        "name), and reason (a brief operational explanation). "
        "Delegate work; do not write patient facts yourself. "
        "Actions: load reads the note; extract lists its facts; write "
        "creates a draft; review checks it; revise corrects a reviewed "
        "draft; save writes it; escalate stops for the instructor. "
        "If facts exist but the report is empty, write. If a report "
        "exists but the review is null, review. "
        "'Not provided' is valid information, not a gap to invent. "
        "After failed review, choose extract if facts need repair "
        "or revise if the draft needs correction. Use the original "
        "note as evidence. Save only after PASS. Escalate when "
        "the task cannot be completed within the remaining steps.",
        json.dumps({"state": state, "allowed": allowed, "steps_left": steps_left}),
        json_output=schema,
    )
    decision = json.loads(raw)
    if decision.get("action") not in allowed:
        raise ValueError(f"Unavailable manager action: {decision.get('action')!r}")
    return decision

print("choose_next function initialized.")

choose_next function initialized.


## Step 4: Dispatcher with Stale Review Invalidation

> **CRITICAL SAFETY RULE:** Any action that changes the report or facts (`extract`, `write`, `revise`) **MUST set `state["review"] = None`**. Otherwise, a stale `PASS` from an earlier draft could be used to prematurely save an unreviewed modification!

In [6]:
def dispatch(action, state, folder, inject_error):
    if action == "load":
        state["note"] = load_patient_note()
    elif action == "extract":
        state["facts"] = extract_facts(state["note"])
        state["report"] = ""
        state["review"] = None
    elif action == "write":
        state["report"] = write_report(state["facts"])
        if inject_error and not state["error_inserted"]:
            state["report"] += KNOWN_ERROR
            state["error_inserted"] = True
        state["review"] = None
    elif action == "review":
        state["review"] = review_report(state["note"], state["report"])
    elif action == "revise":
        state["report"] = revise_report(
            state["note"], state["report"], state["review"]["issues"]
        )
        state["review"] = None  # Crucial: invalidate stale review!
    elif action == "save":
        state["saved_path"] = save_report(state, folder)

print("Dispatcher initialized with stale review invalidation.")

Dispatcher initialized with stale review invalidation.


## Step 5: The Bounded Orchestration Loop (`run_manager`)

In [7]:
def run_manager(folder, inject_error=False, max_steps=MAX_STEPS):
    global CALL_COUNT
    CALL_COUNT = 0
    start = time.perf_counter()
    state = {
        "note": "", "facts": "", "report": "",
        "review": None, "error_inserted": False
    }
    trace = []
    status = "needs_instructor"
    stop_reason = "Step budget exhausted"
    
    for step in range(1, max_steps + 1):
        allowed = allowed_actions(state)
        event = {"step": step, "allowed": allowed}
        trace.append(event)
        try:
            steps_left = max_steps - step + 1
            decision = choose_next(state, allowed, steps_left)
            event["manager"] = decision
            action = decision["action"]
            
            # Code-level double check
            if action not in allowed:
                raise ValueError(f"Action '{action}' rejected by dispatcher")
            if action == "escalate":
                stop_reason = decision.get("reason", "Escalated to instructor")
                event["outcome"] = "Escalated"
                break
                
            dispatch(action, state, folder, inject_error)
            event["outcome"] = "Executed"
            event["state_after"] = json.loads(json.dumps(state))
            
            if action == "save":
                status = "ready_for_human_review"
                stop_reason = "Reviewed classroom draft saved"
                break
        except Exception as exc:
            stop_reason = f"{type(exc).__name__}: {exc}"
            event["error"] = stop_reason
            break
            
    return {
        "model": MODEL_NAME,
        "status": status,
        "stop_reason": stop_reason,
        "state": state,
        "trace": trace,
        "model_requests": CALL_COUNT,
        "elapsed_seconds": round(time.perf_counter() - start, 2),
    }

print("run_manager bounded orchestrator initialized.")

run_manager bounded orchestrator initialized.


## Test 1: Clean Manager Run

**Expected Sequence:**
`load → extract → write → review → save` (5 actions, 8 model calls total: 5 decisions + 3 worker tasks).

In [8]:
clean_folder = Path("outputs") / f"manager_clean_{uuid4().hex[:6]}"
print("Running clean manager workflow...")
clean_record = run_manager(clean_folder, inject_error=False)

print(f"\nStatus:         {clean_record['status']}")
print(f"Stop Reason:    {clean_record['stop_reason']}")
print(f"Model Requests: {clean_record['model_requests']}")
print(f"Steps Taken:    {len(clean_record['trace'])}")

print("\nExecution Trace:")
for ev in clean_record["trace"]:
    step = ev["step"]
    action = ev.get("manager", {}).get("action", "unknown")
    reason = ev.get("manager", {}).get("reason", "")
    print(f"  Step {step}: Action = {action:8} | Reason: {reason}")

Running clean manager workflow...

Status:         ready_for_human_review
Stop Reason:    Reviewed classroom draft saved
Model Requests: 8
Steps Taken:    5

Execution Trace:
  Step 1: Action = load     | Reason: The note is empty, so loading is required to determine if any facts are present. Without loading, no facts can be extracted or evaluated, and no report can be generated.
  Step 2: Action = extract  | Reason: The note contains clinical facts, but the 'facts' field is empty. Extracting ensures the report workflow has valid input to proceed with drafting or review.
  Step 3: Action = write    | Reason: The report is currently empty, and the facts list exists. According to the workflow, if facts exist but the report is empty, a draft must be written. The original note provides all necessary facts, and no patient details are missing or fabricated. Writing a draft is the required first step to proceed.
  Step 4: Action = review   | Reason: The report exists and matches all facts lis

## Test 2: Injected Error & Adaptive Manager Recovery

**Expected Sequence:**
`load → extract → write (error injected) → review (REVISE) → revise → review (PASS) → save`  
(7 actions, 12 model calls total: 7 decisions + 5 worker tasks).

In [9]:
repair_folder = Path("outputs") / f"manager_repair_{uuid4().hex[:6]}"
print("Running manager workflow with DELIBERATE ERROR...")
repair_record = run_manager(repair_folder, inject_error=True)

print(f"\nStatus:         {repair_record['status']}")
print(f"Stop Reason:    {repair_record['stop_reason']}")
print(f"Model Requests: {repair_record['model_requests']}")
print(f"Steps Taken:    {len(repair_record['trace'])}")

print("\nRecovery Execution Trace:")
for ev in repair_record["trace"]:
    step = ev["step"]
    action = ev.get("manager", {}).get("action", "unknown")
    reason = ev.get("manager", {}).get("reason", "")
    print(f"  Step {step}: Action = {action:8} | Reason: {reason}")

# Save trace to disk
trace_path = repair_folder / "trace.json"
repair_folder.mkdir(parents=True, exist_ok=True)
trace_path.write_text(json.dumps(repair_record, indent=2), encoding="utf-8")
print(f"\nSaved trace to: {trace_path}")

Running manager workflow with DELIBERATE ERROR...

Status:         ready_for_human_review
Stop Reason:    Reviewed classroom draft saved
Model Requests: 12
Steps Taken:    7

Recovery Execution Trace:
  Step 1: Action = load     | Reason: The note is empty, so loading is required to determine if any facts are present. Without loading, no facts can be extracted or evaluated, and no report can be generated.
  Step 2: Action = extract  | Reason: The note contains clinical facts, but the 'facts' field is empty. Extracting ensures the report workflow has valid input to proceed with drafting or review.
  Step 3: Action = write    | Reason: The report is currently empty, and the facts list exists. According to the workflow, if facts exist but the report is empty, a draft must be written. The original note provides all necessary facts, and no patient details are missing or fabricated. Writing a draft is the required first step to proceed.
  Step 4: Action = review   | Reason: The report contai

## Test 3: Budget Limit Test (`MAX_STEPS = 2`)

> **Testing Principle:** Temporarily set `max_steps = 2`. A normal run requires at least 5 steps to reach `save`. Verify that Python safely halts execution when the budget is exhausted, terminating with status `"needs_instructor"`.

In [10]:
budget_folder = Path("outputs") / "manager_budget_test"
print("Running manager with MAX_STEPS = 2 (Budget Exhaustion Test)...")
budget_record = run_manager(budget_folder, inject_error=False, max_steps=2)

print(f"\nStatus:      {budget_record['status']}")
print(f"Stop Reason: {budget_record['stop_reason']}")
print(f"Steps run:   {len(budget_record['trace'])}")

assert budget_record["status"] == "needs_instructor", "Must halt with needs_instructor!"
assert "Step budget exhausted" in budget_record["stop_reason"], "Must report step budget exhaustion!"
print("\nSUCCESS: Python safely prevented infinite execution when budget was exhausted.")

Running manager with MAX_STEPS = 2 (Budget Exhaustion Test)...

Status:      needs_instructor
Stop Reason: Step budget exhausted
Steps run:   2

SUCCESS: Python safely prevented infinite execution when budget was exhausted.


## Test 4: Reject Premature Saving

> **Testing Principle:** Verify that Python strictly forbids saving a draft that has not received a passing review, regardless of what the LLM proposes.

In [11]:
print("Running Checkpoint 4: Reject Premature Saving Test...")

# 1. Test save_report directly with unreviewed state
unreviewed_state = {
    "note": "Patient ID: DEMO-001...",
    "facts": "Age: 60...",
    "report": "Draft without review",
    "review": None,
}
try:
    save_report(unreviewed_state, Path("outputs/premature_test"))
    print("FAILED: save_report allowed unreviewed draft to save!")
except ValueError as e:
    print(f"SUCCESS: Premature save correctly blocked: {e}")

# 2. Test save_report with failing review
failing_state = {
    "note": "Patient ID: DEMO-001...",
    "facts": "Age: 60...",
    "report": "Flawed draft with stage III cancer",
    "review": {"decision": "REVISE", "issues": ["Invented stage III cancer"]},
}
try:
    save_report(failing_state, Path("outputs/premature_test"))
    print("FAILED: save_report allowed rejected draft to save!")
except ValueError as e:
    print(f"SUCCESS: Flawed draft save correctly blocked: {e}")

# 3. Verify allowed_actions gatekeeper
actions_before_review = allowed_actions(unreviewed_state)
assert "save" not in actions_before_review, "'save' must not be allowed before PASS!"
print(f"SUCCESS: 'save' is excluded from allowed_actions: {actions_before_review}")

Running Checkpoint 4: Reject Premature Saving Test...
SUCCESS: Premature save correctly blocked: A reviewed PASS is required before saving.
SUCCESS: Flawed draft save correctly blocked: A reviewed PASS is required before saving.
SUCCESS: 'save' is excluded from allowed_actions: ['review', 'escalate']


## Exercise 5 Checkpoint & Final Workshop Synthesis

### Comprehensive 5-Architecture Synthesis Matrix

| Architectural Dimension | Sequential Python (Ex 1) | CrewAI (Ex 2) | AutoGen (Ex 3) | Bounded Revision (Ex 4) | LLM Manager (Ex 5) |
|---|---|---|---|---|---|
| **Control Flow & Topology** | Hardcoded procedural pipeline | Declarative sequential Task DAG | Asynchronous message passing | Bounded conditional branch loop | Dynamic State Machine with LLM Planner |
| **Agent / Worker Abstraction** | Plain Python functions | `Agent(role, goal, backstory)` | `AssistantAgent` | Specialized Python functions | Central Manager LLM + Tool functions |
| **Information Handoff** | Explicit Python variables | Framework `context=[...]` DAG | Explicit variables via async messages | Python variables | Shared `state` dictionary |
| **Error Detection** | Programmatic JSON schema review | Unstructured text review | Unstructured text review | Programmatic JSON schema review | Schema-validated JSON review |
| **Self-Correction Strategy** | None (inspection check only) | None (forward-only pass) | None (forward-only pass) | Exactly 1 bounded revision branch | Adaptive repair (choose `revise` vs `extract`) |
| **Calls (Clean / Flawed)** | 3 / 3 | ~3 / ~3 | 3 / 3 | 3 / 5 | 8 / 12 |
| **Loop Safety Guarantee** | Guaranteed termination (linear code) | Guaranteed termination (DAG) | Guaranteed termination (linear coroutines) | Hardcoded `revisions <= 1` bound | Deterministic `MAX_STEPS = 10` countdown |
| **Industry Sweet-Spot** | High-throughput, deterministic microservices | Content creation & roleplaying DAGs | Conversational multi-turn negotiations | Clinical document QA & self-editing | Dynamic orchestrations with external tools |

### Reflection & Architectural Insights:
1. **Where does the model make a decision vs. where does Python enforce a rule?**  
   *Answer:* The LLM manager plans the next step, chooses an action, and supplies an operational explanation. Python enforces `allowed_actions(state)`, rejects out-of-order calls, invalidates stale reviews on edits, halts when the step budget expires, and forbids saving unapproved drafts.

2. **Did the manager earn its place?**  
   *Answer:* In a clean, linear task, the manager adds significant overhead (8 model requests vs 3). The manager earns its place when the workflow exhibits uncertainty, recovery branches (deciding whether to re-extract or revise), or dynamic tool selection.

3. **Is a fixed sequence autonomous planning?**  
   *Answer:* No. Exercises 1–3 use pre-programmed control flow. True autonomous planning requires the agent to reason about intermediate observations and select actions dynamically, as demonstrated in Exercise 5.